In [40]:
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv
import os

load_dotenv()

con = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    warehouse="COMPUTE_WH",
    database="RIDESHARE",
    schema="MARTS"
)
print("Connected to Snowflake")

Connected to Snowflake


In [41]:
def query(sql):
    cursor = con.cursor()
    cursor.execute(sql)
    cols = [col[0] for col in cursor.description]
    rows = cursor.fetchall()
    return pd.DataFrame(rows, columns=cols)

In [42]:
import plotly.express as px

- - -

## Uber vs Lyft - Volume and Revenue Comparison

#### Which company dominates NYC rideshare, and by how much?

In [43]:
df = query("""
    select
        company,
        count(*) as total_trips,
        round(sum(total_fare), 2) as gross_revenue,
        round(sum(total_fare - driver_pay), 2) as platform_revenue,
        round(avg(total_fare), 2) as avg_fare_per_trip
    from fct_trips
    group by company
""")

df['GROSS_REVENUE'] = df['GROSS_REVENUE'].map('${:,.2f}'.format)
df['PLATFORM_REVENUE'] = df['PLATFORM_REVENUE'].map('${:,.2f}'.format)
df

,COMPANY,TOTAL_TRIPS,GROSS_REVENUE,PLATFORM_REVENUE,AVG_FARE_PER_TRIP
0,Uber,13564036,"$356,808,525.53","$120,105,305.74",26.31
1,Lyft,4898054,"$118,548,815.74","$45,348,420.75",24.20


Analysis shows Uber completely dominating NYC rideshare, accounting for 73% of all rides with riders paying an average of $26.31 per Uber trip compared to $24.20 on Lyft. Although, when it comes to revenue percentage, Lyft keeps a 38% margin while Uber sits at 33.7%. Regardless, Uber is certainly controlling the majority of rides in New York. Brand recognition certainly plays a factor in that success. Uber launched two years before Lyft, expanded internationally far earlier, and evolved beyond rides entirely with Uber Eats — meaning customers don't need to request a ride to stay engaged with the app.

- - -

## Demand Patterns

### 1.
#### What time of the day is busiest?

In [44]:
df = query("""
    select
        to_char(time_from_parts(pickup_hour, 0, 0), 'HH12 AM') as hour_of_day,
        count(*) as trip_volume
    from fct_trips
    group by 1
    order by 2 desc
""")

df['TRIP_VOLUME'] = df['TRIP_VOLUME'].map('{:,}'.format)
df

,HOUR_OF_DAY,TRIP_VOLUME
0,06 PM,"1,155,425"
1,07 PM,"1,122,720"
2,05 PM,"1,085,888"
3,08 PM,"1,019,165"
4,09 PM,"973,566"
5,10 PM,"958,042"
6,04 PM,"950,019"
7,03 PM,"922,560"
8,08 AM,"908,565"
9,02 PM,"896,767"


Data shows late afternoon as the busiest period for rideshare, with 6PM recording the highest volume. This aligns with the post-work commute window, where riders leaving offices opt for a ride over walking or public transit. Demand drops sharply overnight, bottoming out at 4AM, likely a combination of lower demand and fewer drivers on the road.

### 2.
#### Weekday demand compared to weekend demand, by borough

In [45]:
df = query("""
    select
        pickup_borough,
        day_type,
        count(*) as trip_volume,
            round(count(*) * 100.0 / sum(count(*)) over (
                                    partition by day_type
        ), 2) as pct_of_day_week
    from fct_trips
    where pickup_borough not in ('N/A', 'EWR')
    group by 1, 2
    order by 1, 2, 3 desc
""")

df['TRIP_VOLUME'] = df['TRIP_VOLUME'].map('{:,}'.format)
df['PCT_OF_DAY_WEEK'] = df['PCT_OF_DAY_WEEK'].map('{}%'.format)
df

,PICKUP_BOROUGH,DAY_TYPE,TRIP_VOLUME,PCT_OF_DAY_WEEK
0,Bronx,Weekday,"1,889,454",11.81%
1,Bronx,Weekend,"305,986",12.40%
2,Brooklyn,Weekday,"4,148,936",25.94%
3,Brooklyn,Weekend,"631,420",25.59%
4,Manhattan,Weekday,"6,620,147",41.39%
5,Manhattan,Weekend,"952,216",38.59%
6,Queens,Weekday,"3,121,016",19.51%
7,Queens,Weekend,"542,395",21.98%
8,Staten Island,Weekday,"214,296",1.34%
9,Staten Island,Weekend,"35,234",1.43%


Weekdays generate significantly more total trips, consistent with rideshare serving daily work commutes. The more interesting finding, however, is how the *share* of trips per borough remains nearly identical between weekdays and weekends. Manhattan accounts for 41% of weekday trips and 39% of weekend trips — a negligible shift. This structural consistency suggests riders depend on rideshare year-round regardless of day type, not just for commuting.

Note: EWR (Newark Airport) was excluded from this analysis. Although included in the TLC zone lookup as a technical zone, it sits in New Jersey and NYC rideshare companies have extremely limited pickup authority there. Excluding it maintains geographic integrity.

- - -

## Route Profitability

### 1.
#### Revenue per mile for pickup/dropoff borough pairs

In [46]:
df = query("""
        select
            pickup_borough,
            dropoff_borough,
            round(avg((total_fare + tips) / trip_miles), 2) as rev_per_mile,
            round(avg((total_fare + tips) / 
                nullif(trip_duration_minutes, 0) * 60), 2) as rev_per_hour,
            count(*) as total_trips
        from fct_trips
        where pickup_borough not in ('N/A', 'EWR')
            and dropoff_borough not in ('N/A', 'EWR')
        group by 1, 2
        order by 3 desc
""")

df['TOTAL_TRIPS']=df['TOTAL_TRIPS'].map('{:,}'.format)
df['REV_PER_MILE']=df['REV_PER_MILE'].map('${}'.format)
df['REV_PER_HOUR']=df['REV_PER_HOUR'].map('${}'.format)
df

,PICKUP_BOROUGH,DROPOFF_BOROUGH,REV_PER_MILE,REV_PER_HOUR,TOTAL_TRIPS
0,Manhattan,Manhattan,$11.05,$100.71,"5,480,163"
1,Brooklyn,Brooklyn,$7.72,$77.76,"3,632,893"
2,Bronx,Bronx,$6.91,$78.26,"1,613,122"
3,Queens,Queens,$6.77,$85.44,"2,354,542"
4,Manhattan,Brooklyn,$6.33,$92.01,"635,905"
5,Brooklyn,Manhattan,$6.26,$87.57,"575,166"
6,Staten Island,Staten Island,$6.14,$90.12,"213,981"
7,Manhattan,Queens,$5.5,$111.75,"640,226"
8,Queens,Manhattan,$5.38,$103.36,"565,922"
9,Manhattan,Bronx,$5.07,$81.6,"374,592"


Internal trips within dense boroughs like Manhattan generate the highest revenue per mile ($11.05), but congestion erodes that advantage when measured by time. Manhattan to Staten Island routes yield 34% more revenue per hour ($134.59 vs $100.71) despite ranking near the bottom on revenue per mile. Outer-borough highway routes — faster and less traffic-dependent — appear undervalued by traditional per-mile pricing models.

Drivers optimizing for earnings may benefit from prioritizing fewer, faster outer-borough trips over high-volume Manhattan grinding.

### 2.
#### Demand vs Average fare price (by borough)

In [47]:
df = query("""
        select
            pickup_borough,
            round(avg(total_fare), 2) as avg_fare,
            count(*) as total_trips,
            (select
                round(avg(total_fare), 2)
             from fct_trips) as total_avg_fare
        from fct_trips
        where pickup_borough not in ('N/A', 'EWR')
        group by 1
        order by 2 asc
""")

df['AVG_FARE']=df['AVG_FARE'].map('${}'.format)
df['TOTAL_TRIPS']=df['TOTAL_TRIPS'].map('{:,}'.format)
df['TOTAL_AVG_FARE']=df['TOTAL_AVG_FARE'].map('${}'.format)
df

,PICKUP_BOROUGH,AVG_FARE,TOTAL_TRIPS,TOTAL_AVG_FARE
0,Bronx,$19.38,"2,195,440",$25.75
1,Brooklyn,$21.79,"4,780,356",$25.75
2,Staten Island,$22.23,"249,530",$25.75
3,Queens,$28.09,"3,663,411",$25.75
4,Manhattan,$29.08,"7,572,363",$25.75


Brooklyn and the Bronx represent high-demand boroughs with below-average fares — $21.79 and $19.38 respectively, against an overall average of $25.75. Together they account for nearly 37% of all trips while being consistently underpriced relative to the market. Queens, by contrast, averages $28.09 per 
trip despite generating fewer rides than Brooklyn, likely driven by longer average trip distances, particularly rides originating near JFK Airport. The Bronx and Brooklyn data points to pricing power that isn't currently being captured.

- - -

## Tip Behavior

### 1.
#### Which company's riders tip more on average?

In [49]:
df = query("""
    with cte as (
    select
        company,
        round(avg(case when tips > 0 then tips end), 2) as avg_tip,
        sum(case when tips > 0 then 1 else 0 end) as riders_that_tipped,
        count(*) as total_riders
    from fct_trips
    where pickup_borough not in ('N/A', 'EWR')
        and dropoff_borough not in ('N/A', 'EWR')
    group by company
    )
    select
        *,
        round(riders_that_tipped * 100.0 / total_riders, 2) as pct_of_tippers
    from cte
""")

df['AVG_TIP']=df['AVG_TIP'].map('${}'.format)
df['PCT_OF_TIPPERS']=df['PCT_OF_TIPPERS'].map('{}%'.format)
df['RIDERS_THAT_TIPPED']=df['RIDERS_THAT_TIPPED'].map('{:,}'.format)
df['TOTAL_RIDERS']=df['TOTAL_RIDERS'].map('{:,}'.format)
df

,COMPANY,AVG_TIP,RIDERS_THAT_TIPPED,TOTAL_RIDERS,PCT_OF_TIPPERS
0,Lyft,$4.77,"875,102","4,686,564",18.67%
1,Uber,$4.48,"2,493,097","12,936,376",19.27%


Across both platforms, roughly 1 in 5 riders leave a tip , a consistently low rate that suggests most riders view the fare as the full transaction. When tips are given, Lyft riders tip slightly more on average ($4.77 vs $4.48), despite Lyft having significantly fewer total trips. The near-identical 
tipping rates between platforms (19.27% vs 18.67%) suggest tipping behavior is driven by the individual rider rather than platform choice.

### 2.
#### Tips by borough

In [50]:
df = query("""
with cte as (
select
    pickup_borough as borough,
    round(avg(case when tips > 0 then tips end), 2) as avg_tip,
    sum(case when tips > 0 then 1 else 0 end) as riders_that_tipped,
    count(*) as total_riders
from fct_trips
where pickup_borough not in ('N/A', 'EWR')
group by borough
)
select
    *,
    round(riders_that_tipped * 100.0 / total_riders, 2) as pct_of_tippers
from cte
order by avg_tip desc
""")

df['AVG_TIP']=df['AVG_TIP'].map('${}'.format)
df['PCT_OF_TIPPERS']=df['PCT_OF_TIPPERS'].map('{}%'.format)
df['RIDERS_THAT_TIPPED']=df['RIDERS_THAT_TIPPED'].map('{:,}'.format)
df['TOTAL_RIDERS']=df['TOTAL_RIDERS'].map('{:,}'.format)
df

,BOROUGH,AVG_TIP,RIDERS_THAT_TIPPED,TOTAL_RIDERS,PCT_OF_TIPPERS
0,Queens,$5.83,"697,626","3,663,411",19.04%
1,Manhattan,$5.26,"1,817,808","7,572,363",24.01%
2,Staten Island,$4.4,"46,916","249,530",18.80%
3,Brooklyn,$4.2,"843,257","4,780,356",17.64%
4,Bronx,$3.92,"171,983","2,195,440",7.83%


Manhattan riders tip most frequently at 24%, likely reflecting the borough's higher-income demographic and concentration of business travel. Queens riders tip less often (19%) but leave the highest average tip amount at $5.83, possibly reflecting longer, higher-fare airport-adjacent trips where riders feel more inclined to tip generously.

The standout finding is the Bronx, where only 7.83% of riders leave a tip — less than a third of Manhattan's rate. For drivers, this makes the Bronx the weakest borough for tip income, compounding its already below-average fare rates seen in the route profitability analysis.

- - -

## Driver Economics

### 1.
#### What percentage of total fare goes to drivers on Uber vs Lyft?

In [53]:
df = query("""
select
    company,
    round(avg(driver_pay * 100.0 / total_fare), 2) as pct_to_driver
from fct_trips
group by company
order by 2 desc
""")

df['PCT_TO_DRIVER']=df['PCT_TO_DRIVER'].map('{}%'.format)
df

,COMPANY,PCT_TO_DRIVER
0,Uber,68.23%
1,Lyft,63.02%


Uber directs a larger share of each fare to drivers — 68.23% compared to Lyft's 63.02%, a difference of 5.21 percentage points. For drivers weighing which platform to prioritize, this is a meaningful distinction: on an average $25 fare, an Uber driver takes home roughly $1.30 more than a Lyft driver for the same ride.

### 2.
#### Most profitable boroughs for drivers

In [54]:
query("""
select
    pickup_borough as borough,
    round(avg(driver_pay / trip_miles), 2) as avg_pay_per_mile,
    round(avg(driver_pay / 
        nullif(trip_duration_minutes, 0) * 60), 2) as avg_pay_per_hour
from fct_trips
where pickup_borough not in ('N/A', 'EWR')
group by borough
order by 2 desc
""")

,BOROUGH,AVG_PAY_PER_MILE,AVG_PAY_PER_HOUR
0,Manhattan,5.43,57.72
1,Brooklyn,4.82,53.18
2,Bronx,4.46,55.68
3,Staten Island,4.05,63.67
4,Queens,4.02,58.27


Manhattan leads on driver pay per mile ($5.43) but ranks last on pay per hour ($57.72). Staten Island inverts this entirely, lowest per-mile pay ($4.05) but highest per hour ($63.67). Dense urban pricing inflates the per-mile rate, but traffic neutralizes it. A driver completing two efficient Staten Island 
trips in an hour earns more than one grinding through Manhattan traffic for the same period.